# Практическое задание 2: Улучшение качества LLM классификатора

In [36]:
from IPython.display import clear_output
!pip install pandas mistralai tqdm
clear_output()

Файл с данными можно скачать по ссылке: https://disk.yandex.ru/d/ErrpfkgFkdtOIQ.  
Подгружаем данные:

In [37]:
import pandas as pd

data = pd.read_csv("tweets_coronavirus_final.csv")

Настраиваем LLM API:

Переходим на [сайт](https://console.mistral.ai/home). Далее регистрируемся на нем, переходим в API Keys и создаем новый ключ. Его мы будем использовать для авторизации.

Далее, если вы работаете в Google Colab, нужно добавить созданный ключ в Секреты. Для этого откройте соответствующую панель в меню слева и нажмите "Добавить секрет". В поле "Название" введите `MISTRAL_API_KEY`, а в поле "Значение" вставьте ваш ключ. Разрешите доступ из блокнотов. Если же вы работаете в локальной среде, то добавьте созданный ключ в переменные окружения.

In [38]:
import os
from pathlib import Path
from dotenv import load_dotenv
from mistralai.client import Mistral

# Загружаем переменные из файла .env
load_dotenv(Path.cwd() / ".env", override=True)

# Получаем API-ключ
API_KEY = os.getenv("MISTRAL_API_KEY")

if not API_KEY:
    raise ValueError("Не найден MISTRAL_API_KEY в .env")

llm_client = Mistral(api_key=API_KEY)

Посмотрим на наши данные:

In [44]:
data.head()

,UserName,ScreenName,Location,TweetAt,OriginalTweet,Sentiment
0,8041,52993,"Livingston County, NY",18-03-2020,COVID-19 PRICE GOUGING HOTLINE: If you see exc...,0
1,22726,67678,"Adelaide, Australia",24-03-2020,URGENT PLEASE SHARE Hearing reports of fatalit...,0
2,36557,81509,"McCool, MS",07-04-2020,Taste of Power cuts prices with 50 while donat...,1
3,17518,62470,NaN,21-03-2020,Just been to supermarket to get a few things. ...,0
4,17470,62422,Morgantown WV,21-03-2020,@jacknicas Right...he had no idea that Covid-1...,0


In [43]:
print(data.OriginalTweet[:5].to_markdown())

|    | OriginalTweet                                                                                                                                                                                                                                                                                                                                                                                       |
|---:|:----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
|  0 | COVID-19 PRICE GOUGING HOTLINE: If you see excessively priced consumer goods and services that are used primarily for personal, family or household purposes to prevent or respond to the COVID-19 viru

In [42]:
data.Sentiment.value_counts()

Sentiment
0    59
1    41
Name: count, dtype: int64

Наша задача &mdash; предсказывать сентимент твита. Закодируем в числа наш таргет:

In [39]:
def text2label(text: str) -> int:
    if text in ("Positive", "Extremely Positive"):
        return 1
    else:
        return 0

In [40]:
data["Sentiment"] = data["Sentiment"].apply(text2label)

In [41]:
data.Sentiment.value_counts()

Sentiment
0    59
1    41
Name: count, dtype: int64

Определим модель, промпты и параметры генерации LLM:

In [45]:
MODEL_NAME = "ministral-3b-2410"  # вы можете попробовать другие модели https://docs.mistral.ai/getting-started/models/models_overview/

SYSTEM_PROMPT = "Please classify the user's text. Respond with only one word: 'positive' or 'negative'. No other words allowed."

USER_PROMPT = "{}"

SAMPLING_PARAMETERS = {"temperature": 0.1}  # вы можете попробовать поизменять параметры, чтобы добиться лучшего эффекта https://docs.mistral.ai/guides/sampling/

Функция для обработки ответа модели. Здесь вы переводите ответ модели в метку класса (0 или 1):

In [46]:
def response2label(response: str) -> int:
    response = response.strip().lower()

    if response == "positive":
        return 1
    elif response in ("negative", "neutral"):
        return 0
    else:
        raise ValueError(f"Неожиданный ответ модели: {response!r}")

Код для быстрой обработки датасета с помощью LLM:

In [47]:
import asyncio
from sklearn.metrics import accuracy_score
from tenacity import retry, stop_after_attempt, wait_fixed
from tqdm.asyncio import tqdm_asyncio


# классифицируем один твит с помощью LLM, если API вернёт ошибку, повторяем запрос до 3 раз с паузой 20 секунд
@retry(stop=stop_after_attempt(3), wait=wait_fixed(20), reraise=True)
async def classify_sentiment(text):

    # отправляем модели инструкцию и текст
    chat_response = await llm_client.chat.complete_async(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": USER_PROMPT.format(text)},
        ],
        **SAMPLING_PARAMETERS
    )
    # пол. ответ, преобразуем в 0 или 1
    return response2label(chat_response.choices[0].message.content)

# классифицируем сразу несколько твитов
async def classify_batch(texts, delay=2):
    
    # запускаем классификацию для каждого твита с паузой между запросами
    tasks = []
    for text in texts:
        task = asyncio.create_task(classify_sentiment(text))
        tasks.append(task)
        await asyncio.sleep(delay)
    
    # Дожидаемся результатов всех запросов
    predictions = await tqdm_asyncio.gather(*tasks)
    return predictions

In [ ]:
# Предсказания
y_pred = await classify_batch(data['OriginalTweet'])

# Истинные метки
y_true = data["Sentiment"]

# Метрика
acc = accuracy_score(y_true, y_pred)
print("Accuracy:", acc)

100%|██████████| 100/100 [00:00<00:00, 56656.81it/s]

Accuracy: 0.61


Далее вам предстоит улучшать качество. Этого можно добиться через тюнинг промптов, параметров генерации и выбором подходящей модели.

Enjoy!

## Эксперименты по улучшению качества

Перед экспериментами немного изменили обработку ответов модели: добавили удаление лишних пробелов и приведение текста к нижнему регистру. Также учли, что модель может вернуть neutral (воспринимаем их как negative и относим к классу 0), а positive к классу 1. Если модель вернёт что-то другое, возникнет ошибка, которая позволит увидеть некорректный ответ.

### Эксперимент 1. Замена модели

Проверяем, улучшит ли более новая модель `ministral-8b-2512` точность классификации по сравнению с `ministral-3b-2410`. Исходные промпты и параметры генерации оставляем без изменений. Для сравнения используем сохранённую Accuracy = 0.61 на тех же 100 твитах.

In [19]:
MODEL_NAME = "ministral-8b-2512"


In [44]:
model_v2_pred = await classify_batch(data["OriginalTweet"])

100%|██████████| 100/100 [00:00<00:00, 93811.32it/s]


In [45]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

model_v2_acc = accuracy_score(data["Sentiment"], model_v2_pred)
# Precision, Recall и F1 считаем для положительного класса (1).
model_v2_precision = precision_score(data["Sentiment"], model_v2_pred, zero_division=0)
model_v2_recall = recall_score(data["Sentiment"], model_v2_pred, zero_division=0)
model_v2_f1 = f1_score(data["Sentiment"], model_v2_pred, zero_division=0)

print("Исходная модель ministral-3b-2410: Accuracy = 0.61 (сохранённый результат)")
print(f"Новая модель {MODEL_NAME}:")
print(f"Accuracy = {model_v2_acc:.4f}")
print(f"Precision = {model_v2_precision:.4f}")
print(f"Recall = {model_v2_recall:.4f}")
print(f"F1-score = {model_v2_f1:.4f}")


Исходная модель ministral-3b-2410: Accuracy = 0.61 (сохранённый результат)
Новая модель ministral-8b-2512:
Accuracy = 0.9000
Precision = 0.8974
Recall = 0.8537
F1-score = 0.8750


In [46]:
model_v2_errors = pd.DataFrame({
    "Текст твита": data["OriginalTweet"],
    "Правильная метка": data["Sentiment"],
    "Предсказанная метка": model_v2_pred,
})
model_v2_errors = model_v2_errors[
    model_v2_errors["Правильная метка"] != model_v2_errors["Предсказанная метка"]
].reset_index(drop=True)

model_v2_errors


,Текст твита,Правильная метка,Предсказанная метка
0,If I had a blue badge @Twitter account and the...,0,1
1,Is it gonna look really good on my resume that...,1,0
2,Use Online Grocery Shopping and order before s...,1,0
3,While the US states are outbidding each other ...,1,0
4,Let me stock food in time. Addressing the Nati...,0,1
5,You can tell who was and who wasn't in a March...,1,0
6,#PardeeProfs\r\r\nHow does #Coronavirus impact...,0,1
7,Soap vs hand sanitizer What is it Professor ...,1,0
8,"""The wheat complex soared this morning amid st...",1,0
9,This #coronavirus got these next day/less than...,0,1


### Выводы по эксперименту 1

После замены `ministral-3b-2410` на `ministral-8b-2512` Accuracy увеличилась с 0.61 до 0.90 (исходные промпты и параметры генерации не менялись).

Для новой модели Precision составила 0.8974, Recall - 0.8537, F1-score - 0.8750. Модель ошиблась на 10 из 100 твитов.

Таким образом, замена модели заметно повысила точность на рассматриваемой выборке. Далее попробуем улучшить результат за счёт изменения промпта.

### Эксперимент 2. Замена промта

Для начала проанализируем тексты твиттов, где модель ошиблась с предсказанием.

In [52]:
from IPython.display import display, HTML
import html

for _, row in model_v2_errors.iterrows():
    display(HTML(f"""
    <div style="margin-bottom:16px; padding:12px; border:1px solid #888; border-radius:8px;">
        <p style="white-space:pre-wrap; overflow-wrap:anywhere;">
            {html.escape(str(row['Текст твита']))}
        </p>
        <b>Правильная метка:</b> {row['Правильная метка']}
        &nbsp;&nbsp;
        <b>Предсказание:</b> {row['Предсказанная метка']}
    </div>
    """))

Вывод: анализ ошибок показал, что модель иногда неверно классифицирует твиты со смешанной тональностью (например, твит №4), иронией (например, твиты №2, №6) и с отсутствием эмоций (например, твитт №7). 

Попробуем уточнить системный промпт: подробнее опишем оба класса и попросим модель учитывать общий смысл сообщения, а не отдельные эмоционально окрашенные слова. Модель и параметры генерации оставим без изменений.

In [53]:
SYSTEM_PROMPT = """
You are a sentiment classifier for tweets about COVID-19.

Classify each tweet into one of two categories:

positive:
The tweet expresses positive or extremely positive sentiment,
including praise, gratitude, approval, optimism, or satisfaction.

negative:
The tweet expresses negative, extremely negative, or neutral sentiment.
This includes complaints, criticism, fear, frustration, and purely
informational statements without clear positive sentiment.

Consider the overall meaning of the tweet, including context,
sarcasm, humor, and mixed emotions. Do not classify a tweet
as positive solely because it contains positive words.

If the tweet contains both positive and negative elements,
identify the author's main sentiment.

Respond with exactly one word: positive or negative.
"""


In [54]:
prompt_v2_pred = await classify_batch(data["OriginalTweet"])

prompt_v2_acc = accuracy_score(data["Sentiment"], prompt_v2_pred)
prompt_v2_precision = precision_score(data["Sentiment"], prompt_v2_pred, zero_division=0)
prompt_v2_recall = recall_score(data["Sentiment"], prompt_v2_pred, zero_division=0)
prompt_v2_f1 = f1_score(data["Sentiment"], prompt_v2_pred, zero_division=0)

print(f"Accuracy = {prompt_v2_acc:.4f}")
print(f"Precision = {prompt_v2_precision:.4f}")
print(f"Recall = {prompt_v2_recall:.4f}")
print(f"F1-score = {prompt_v2_f1:.4f}")
print(f"Accuracy эксперимента 1: {model_v2_acc:.4f}")
print(f"Accuracy эксперимента 2: {prompt_v2_acc:.4f}")
print(f"Изменение Accuracy: {prompt_v2_acc - model_v2_acc:+.4f}")


100%|██████████| 100/100 [00:00<00:00, 38112.71it/s]

Accuracy = 0.7400
Precision = 0.9412
Recall = 0.3902
F1-score = 0.5517
Accuracy эксперимента 1: 0.9000
Accuracy эксперимента 2: 0.7400
Изменение Accuracy: -0.1600


### Вывод

После уточнения системного промпта Accuracy снизилась с 0.90 до 0.74. При этом Precision выросла с 0.8974 до 0.9412 (модель стала острожнее выделять positive), а Recall упала с 0.8537 до 0.3902.

Вероятно, новый промпт слишком сильно сместил модель в сторону класса 0: она стала реже ошибочно относить неположительные твиты к положительным, но при этом пропустила гораздо больше действительно положительных сообщений.

Таким образом, более подробная инструкция не улучшила качество классификации. Далее попробуем использовать более короткий и сбалансированный промпт.

In [56]:
SYSTEM_PROMPT = """
Classify the overall sentiment of the tweet.

Positive sentiment, including gratitude, hope,
approval, and satisfaction -> positive.

Negative or neutral sentiment -> negative.

Consider the author's overall attitude, including
positive feelings expressed indirectly.

Respond with exactly one word: positive or negative.
"""


In [57]:
prompt_v3_pred = await classify_batch(data["OriginalTweet"])

prompt_v3_acc = accuracy_score(data["Sentiment"], prompt_v3_pred)
prompt_v3_precision = precision_score(data["Sentiment"], prompt_v3_pred, zero_division=0)
prompt_v3_recall = recall_score(data["Sentiment"], prompt_v3_pred, zero_division=0)
prompt_v3_f1 = f1_score(data["Sentiment"], prompt_v3_pred, zero_division=0)

print(f"Accuracy = {prompt_v3_acc:.4f}")
print(f"Precision = {prompt_v3_precision:.4f}")
print(f"Recall = {prompt_v3_recall:.4f}")
print(f"F1-score = {prompt_v3_f1:.4f}")
print(f"Accuracy эксперимента 1: {globals().get('model_v2_acc', 0.90):.4f}")
print(f"Accuracy эксперимента 2: {globals().get('prompt_v2_acc', 0.74):.4f}")
print(f"Accuracy эксперимента 2 (короткий промпт): {prompt_v3_acc:.4f}")


100%|██████████| 100/100 [00:00<00:00, 40399.77it/s]

Accuracy = 0.8600
Precision = 1.0000
Recall = 0.6585
F1-score = 0.7941
Accuracy эксперимента 1: 0.9000
Accuracy эксперимента 2: 0.7400
Accuracy эксперимента 3: 0.8600


### Выводы по эксперименту 2

После сокращения системного промпта Accuracy выросла с 0.74 до 0.86, а Recall - с 0.3902 до 0.6585. При этом Precision достигла 1.0: все твиты, которые модель определила как положительные, действительно относились к классу 1.

Однако модель по-прежнему пропускает часть положительных твитов, поэтому результат оказался хуже, чем в первом эксперименте (Accuracy = 0.90).

Таким образом, сокращение промпта улучшило качество по сравнению с подробным вариантом инструкции, но исходный промпт пока остаётся наиболее эффективным. Далее попробуем few-shot prompting (добавление примеров классификации в запрос к модели).

### Эксперимент 3. Few-shot prompting

В предыдущем эксперименте изменение системного промпта не позволило превысить Accuracy = 0.90. Поэтому вернём исходную инструкцию и попробуем few-shot prompting.

Добавим в запрос несколько примеров твитов с правильными метками. При выборе примеров учтём проблемы, обнаруженные при анализе ошибок: смешанную тональность, иронию, косвенно выраженную положительную оценку и нейтральные сообщения. Сами ошибочно классифицированные твиты использовать в качестве примеров не будем, чтобы не подгонять модель под проверочную выборку.

Модель и параметры генерации оставим без изменений.

In [20]:
SYSTEM_PROMPT = (
    "Please classify the user's text. "
    "Respond with only one word: 'positive' or 'negative'. "
    "No other words allowed."
)


In [21]:
USER_PROMPT = """Tweet: Thank you to the volunteers who brought groceries while we were isolating. You made a difficult week easier.
Sentiment: positive

Tweet: The pharmacy now delivers during quarantine. That is one less thing keeping me awake at night.
Sentiment: positive

Tweet: My grocery delivery was cancelled again during lockdown, and nobody will explain why. I am fed up.
Sentiment: negative

Tweet: Brilliant, another shopper coughing over the vegetables without a mask. Just what this pandemic needed.
Sentiment: negative

Tweet: The community COVID testing centre opens at 9 am on weekdays. Appointments can be booked online.
Sentiment: negative

Tweet: The lockdown queue was exhausting, but the staff made sure my elderly dad got everything he needed. I left feeling grateful.
Sentiment: positive

Tweet: {}
Sentiment:"""


In [24]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

few_shot_pred = await classify_batch(data["OriginalTweet"])

few_shot_acc = accuracy_score(data["Sentiment"], few_shot_pred)
few_shot_precision = precision_score(data["Sentiment"], few_shot_pred, zero_division=0)
few_shot_recall = recall_score(data["Sentiment"], few_shot_pred, zero_division=0)
few_shot_f1 = f1_score(data["Sentiment"], few_shot_pred, zero_division=0)

print(f"Accuracy = {few_shot_acc:.4f}")
print(f"Precision = {few_shot_precision:.4f}")
print(f"Recall = {few_shot_recall:.4f}")
print(f"F1-score = {few_shot_f1:.4f}")
print("Лучший предыдущий результат: Accuracy = 0.90")
print(f"Few-shot prompting: Accuracy = {few_shot_acc:.4f}")
print(f"Изменение Accuracy: {few_shot_acc - 0.90:+.4f}")

100%|██████████| 100/100 [00:20<00:00,  4.83it/s]

Accuracy = 0.9200
Precision = 0.8837
Recall = 0.9268
F1-score = 0.9048
Лучший предыдущий результат: Accuracy = 0.90
Few-shot prompting: Accuracy = 0.9200
Изменение Accuracy: +0.0200


### Анализ ошибок после few-shot prompting

Посмотрим, в каких твитах модель продолжает ошибаться после добавления few-shot примеров.


In [27]:
from IPython.display import display, HTML
import html

few_shot_errors = data[["OriginalTweet", "Sentiment"]].copy()
few_shot_errors["Предсказание"] = few_shot_pred

few_shot_errors = few_shot_errors[
    few_shot_errors["Sentiment"] != few_shot_errors["Предсказание"]
]

cards = []

for i, row in few_shot_errors.iterrows():
    tweet = html.escape(str(row["OriginalTweet"]))
    true_label = int(row["Sentiment"])
    pred_label = int(row["Предсказание"])

    cards.append(f"""
    <div style="border:1px solid #888; border-radius:10px;
                padding:16px; margin-bottom:12px;">
        <div style="font-weight:bold; margin-bottom:10px;">
            Твит №{i}
        </div>
        <div style="white-space:pre-wrap; overflow-wrap:anywhere;
                    line-height:1.6; margin-bottom:12px;">
            {tweet}
        </div>
        <div>
            Правильная метка: <b>{true_label}</b>
            &nbsp;&nbsp;|&nbsp;&nbsp;
            Предсказание: <b>{pred_label}</b>
        </div>
    </div>
    """)

display(HTML(
    f"<h3>Ошибки few-shot: {len(few_shot_errors)}</h3>"
    + "".join(cards)
))

### Вывод

После few-shot prompting модель ошиблась в 8 твитах из 100. Основные сложности возникли с иронией (№9, №77), юмором (№63) и сообщениями без явной эмоциональной оценки (№74, №94). При этом некоторые правильные метки сами по себе неоднозначны.

Accuracy выросла до 0.92, но ошибки всё ещё остаются. Далее попробуем изменить количество и состав примеров в промпте.

### Изменение few-shot примеров

В предыдущем эксперименте few-shot prompting позволил повысить Accuracy до 0.92. Однако анализ ошибок показал, что модель продолжает путаться в сообщениях с иронией, смешанной тональностью и косвенно выраженной оценкой.

Попробуем изменить количество и состав примеров: вместо шести используем четыре новых примера, охватывающих разные типы эмоциональной окраски. Системный промпт, модель и параметры генерации оставим прежними.


In [49]:
USER_PROMPT = """Tweet: Every evening in quarantine, a hot meal appears at my door from next door. When this is over, their garden is getting a month of free weeding from me.
Sentiment: positive

Tweet: Our cafe lost weeks of business to COVID restrictions, but the first takeaway orders are coming in. We have a long road ahead, and I finally believe we can make it.
Sentiment: positive

Tweet: What a thoughtful pandemic policy: tell sick workers to stay home, then cut their pay if they do. Truly looking after people.
Sentiment: negative

Tweet: The council will publish its weekly COVID case totals on Thursday at noon. Figures will be listed by district.
Sentiment: negative

Tweet: {}
Sentiment:"""


In [31]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
few_shot_v2_pred = await classify_batch(data["OriginalTweet"])

few_shot_v2_acc = accuracy_score(data["Sentiment"], few_shot_v2_pred)
few_shot_v2_precision = precision_score(data["Sentiment"], few_shot_v2_pred, zero_division=0)
few_shot_v2_recall = recall_score(data["Sentiment"], few_shot_v2_pred, zero_division=0)
few_shot_v2_f1 = f1_score(data["Sentiment"], few_shot_v2_pred, zero_division=0)

print(f"Accuracy = {few_shot_v2_acc:.4f}")
print(f"Precision = {few_shot_v2_precision:.4f}")
print(f"Recall = {few_shot_v2_recall:.4f}")
print(f"F1-score = {few_shot_v2_f1:.4f}")
print("Предыдущий few-shot результат: Accuracy = 0.9200")
print(f"Новый few-shot результат: Accuracy = {few_shot_v2_acc:.4f}")
print(f"Изменение Accuracy: {few_shot_v2_acc - 0.92:+.4f}")


100%|██████████| 100/100 [00:00<00:00, 36941.20it/s]

Accuracy = 0.9400
Precision = 0.9070
Recall = 0.9512
F1-score = 0.9286
Предыдущий few-shot результат: Accuracy = 0.9200
Новый few-shot результат: Accuracy = 0.9400
Изменение Accuracy: +0.0200


### Выводы по эксперименту 3

После замены шести few-shot примеров на четыре новых Accuracy выросла с 0.92 до 0.94, а F1-score - с 0.9048 до 0.9286. При этом выросли и Precision (0.9070), и Recall (0.9512).

Таким образом, изменение состава и количества примеров позволило улучшить качество классификации. Лучший результат пока получен на модели `ministral-8b-2512` с исходным системным промптом и четырьмя few-shot примерами.

### Эксперимент 4. Замена модели на Ministral 14B

Попробуем использовать более крупную модель `ministral-14b-2512`.

Чтобы сравнение было корректным, оставим системный и пользовательский промпты, а также параметры генерации без изменений. Таким образом, будем проверять только влияние выбора модели на качество классификации.


In [48]:
MODEL_NAME = "ministral-14b-2512"


In [50]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

model_14b_pred = await classify_batch(data["OriginalTweet"])

model_14b_acc = accuracy_score(data["Sentiment"], model_14b_pred)
model_14b_precision = precision_score(data["Sentiment"], model_14b_pred, zero_division=0)
model_14b_recall = recall_score(data["Sentiment"], model_14b_pred, zero_division=0)
model_14b_f1 = f1_score(data["Sentiment"], model_14b_pred, zero_division=0)

model_14b_comparison = pd.DataFrame({
    "Метрика": ["Accuracy", "Precision", "Recall", "F1-score"],
    "Ministral 8B (4-shot)": [0.9400, 0.9070, 0.9512, 0.9286],
    "Ministral 14B": [model_14b_acc, model_14b_precision, model_14b_recall, model_14b_f1],
})
print(model_14b_comparison.to_string(index=False, float_format="{:.4f}".format))


100%|██████████| 100/100 [00:00<00:00, 182044.44it/s]

  Метрика  Ministral 8B (эксперимент 4)  Ministral 14B
 Accuracy                        0.9400         0.8300
Precision                        0.9070         0.8529
   Recall                        0.9512         0.7073
 F1-score                        0.9286         0.7733


### Выводы по эксперименту 4

При замене `ministral-8b-2512` на `ministral-14b-2512` Accuracy снизилась с 0.94 до 0.83, а F1-score - с 0.9286 до 0.7733.

При этом системный и пользовательский промпты, а также параметры генерации оставались прежними.

Таким образом, увеличение размера модели не привело к улучшению качества классификации. Лучшей конфигурацией на нашей выборке осталась `ministral-8b-2512` с исходным системным промптом и четырьмя few-shot примерами.

### Эксперимент 5. Изменение параметров генерации

Во всех предыдущих экспериментах использовалась `temperature` = 0.1. Попробуем уменьшить её до 0, чтобы проверить, повлияет ли снижение случайности генерации на Accuracy и F1-score.

In [52]:
MODEL_NAME = "ministral-8b-2512"

SYSTEM_PROMPT = (
    "Please classify the user's text. "
    "Respond with only one word: 'positive' or 'negative'. "
    "No other words allowed."
)

USER_PROMPT = """Tweet: Every evening in quarantine, a hot meal appears at my door from next door. When this is over, their garden is getting a month of free weeding from me.
Sentiment: positive

Tweet: Our cafe lost weeks of business to COVID restrictions, but the first takeaway orders are coming in. We have a long road ahead, and I finally believe we can make it.
Sentiment: positive

Tweet: What a thoughtful pandemic policy: tell sick workers to stay home, then cut their pay if they do. Truly looking after people.
Sentiment: negative

Tweet: The council will publish its weekly COVID case totals on Thursday at noon. Figures will be listed by district.
Sentiment: negative

Tweet: {}
Sentiment:"""

SAMPLING_PARAMETERS = {"temperature": 0}


In [53]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
temperature_0_pred = await classify_batch(data["OriginalTweet"])

temperature_0_acc = accuracy_score(data["Sentiment"], temperature_0_pred)
temperature_0_precision = precision_score(data["Sentiment"], temperature_0_pred, zero_division=0)
temperature_0_recall = recall_score(data["Sentiment"], temperature_0_pred, zero_division=0)
temperature_0_f1 = f1_score(data["Sentiment"], temperature_0_pred, zero_division=0)

print(f"Accuracy = {temperature_0_acc:.4f}")
print(f"Precision = {temperature_0_precision:.4f}")
print(f"Recall = {temperature_0_recall:.4f}")
print(f"F1-score = {temperature_0_f1:.4f}")
print("Лучший предыдущий результат: Accuracy = 0.9400, F1-score = 0.9286")
print(f"Изменение Accuracy: {temperature_0_acc - 0.94:+.4f}")
print(f"Изменение F1-score: {temperature_0_f1 - 0.9286:+.4f}")


100%|██████████| 100/100 [00:00<00:00, 45032.25it/s]

Accuracy = 0.9400
Precision = 0.9070
Recall = 0.9512
F1-score = 0.9286
Лучший предыдущий результат: Accuracy = 0.9400, F1-score = 0.9286
Изменение Accuracy: +0.0000
Изменение F1-score: -0.0000


### Выводы по эксперименту 5

При уменьшении temperature с 0.1 до 0 Accuracy осталась равной 0.94, а F1-score - 0.9286. Остальные метрики также не изменились.

Таким образом, уменьшение температуры не улучшило качество классификации на нашей выборке. Оба значения показали одинаковый результат.

# Итоговые выводы

В ходе экспериментов удалось повысить Accuracy с 0.61 до 0.94, а лучший F1-score составил 0.9286.

Лучшая конфигурация:
- **Модель:** `ministral-8b-2512`
- **System prompt:** исходный
- **User prompt:** 4 few-shot примера
- **Temperature:** `0.1` или `0` (одинаковые метрики)

Основное улучшение дала замена модели, а few-shot prompting позволил дополнительно повысить качество. Изменение системного промпта и переход на более крупную Ministral 14B оказались менее эффективными, а уменьшение temperature с 0.1 на 0 не повлияло на качество.

Таким образом, выбранная конфигурация показала лучшие Accuracy и F1-score среди проверенных вариантов.

#  Параметры:

In [54]:
print("Модель:", MODEL_NAME)
print("System prompt:", SYSTEM_PROMPT)
print("User prompt:\n", USER_PROMPT)
print("Количество few-shot примеров:", USER_PROMPT.count("Tweet: ") - 1)
print("Параметры генерации:", SAMPLING_PARAMETERS)

Модель: ministral-8b-2512
System prompt: Please classify the user's text. Respond with only one word: 'positive' or 'negative'. No other words allowed.
User prompt:
 Tweet: Every evening in quarantine, a hot meal appears at my door from next door. When this is over, their garden is getting a month of free weeding from me.
Sentiment: positive

Tweet: Our cafe lost weeks of business to COVID restrictions, but the first takeaway orders are coming in. We have a long road ahead, and I finally believe we can make it.
Sentiment: positive

Tweet: What a thoughtful pandemic policy: tell sick workers to stay home, then cut their pay if they do. Truly looking after people.
Sentiment: negative

Tweet: The council will publish its weekly COVID case totals on Thursday at noon. Figures will be listed by district.
Sentiment: negative

Tweet: {}
Sentiment:
Количество few-shot примеров: 4
Параметры генерации: {'temperature': 0}
